# Transformer Architecture

Comprehensive guide to Transformers with PyTorch and HuggingFace transformers library.

📺 **Video Lecture:** [https://youtu.be/yPTi4Ot5qoM](https://youtu.be/yPTi4Ot5qoM)

## Prerequisites

```bash
pip install torch transformers numpy matplotlib
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

## 1. Scaled Dot-Product Attention

The core mechanism of Transformers.
Formula: Attention(Q, K, V) = softmax(QK^T / √d_k)V

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention
    Args:
        Q: Query matrix (batch_size, seq_len, d_k)
        K: Key matrix (batch_size, seq_len, d_k)
        V: Value matrix (batch_size, seq_len, d_v)
        mask: Optional attention mask
    Returns:
        output: Attention output
        attention_weights: Attention weights
    """
    d_k = Q.shape[-1]
    
    # Compute attention scores
    scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
    
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # Apply softmax
    attention_weights = F.softmax(scores, dim=-1)
    
    # Apply attention to values
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# Example: Create dummy tensors
batch_size, seq_len, d_k = 2, 4, 64
Q = torch.randn(batch_size, seq_len, d_k)
K = torch.randn(batch_size, seq_len, d_k)
V = torch.randn(batch_size, seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"Attention weights sum (should be ~1): {weights[0, 0].sum().item():.4f}")

## 2. Multi-Head Attention

Allows the model to attend to different representation subspaces.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, Q, K, V, mask=None):
        batch_size = Q.shape[0]
        
        # Linear transformations and reshape for multi-head
        Q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        # Final linear transformation
        output = self.W_o(attn_output)
        
        return output, attn_weights

# Example usage
d_model, num_heads = 256, 8
mha = MultiHeadAttention(d_model, num_heads)
X = torch.randn(2, 10, d_model)  # batch_size=2, seq_len=10, d_model=256
output, weights = mha(X, X, X)
print(f"MHA output shape: {output.shape}")
print(f"Total parameters: {sum(p.numel() for p in mha.parameters())}")

In [ ]:
# PyTorch's built-in MultiheadAttention
mha_pytorch = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, batch_first=True)
X_test = torch.randn(2, 10, d_model)
output_pytorch, weights_pytorch = mha_pytorch(X_test, X_test, X_test)
print(f"PyTorch MHA output shape: {output_pytorch.shape}")
print(f"PyTorch MHA attention weights shape: {weights_pytorch.shape}")

## 3. Positional Encoding

Sinusoidal positional encoding to capture sequence position information.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len=1000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x):
        # x: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

# Visualize positional encoding
pe_encoder = PositionalEncoding(d_model=256, max_seq_len=100)
pe_matrix = pe_encoder.pe[0, :50].numpy()  # First 50 positions

plt.figure(figsize=(12, 4))
plt.imshow(pe_matrix, cmap='viridis', aspect='auto')
plt.colorbar(label='Encoding Value')
plt.xlabel('Dimension')
plt.ylabel('Sequence Position')
plt.title('Sinusoidal Positional Encoding Heatmap')
plt.tight_layout()
plt.show()

print(f"Positional encoding shape: {pe_matrix.shape}")

## 4. Transformer Encoder Block

Core building block: Multi-head attention + Feed-forward network.

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, src_mask=None):
        # Multi-head attention with residual connection
        attn_output, _ = self.mha(x, x, x, attn_mask=src_mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # Feed-forward with residual connection
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_output))
        
        return x

# Example: Create encoder block and pass data
encoder_block = TransformerEncoderBlock(d_model=256, num_heads=8, d_ff=512)
x = torch.randn(2, 10, 256)
output = encoder_block(x)
print(f"Encoder block output shape: {output.shape}")
print(f"Total parameters: {sum(p.numel() for p in encoder_block.parameters())}")

## 5. Load Pretrained BERT

Use HuggingFace transformers to load a pretrained BERT model.

In [ ]:
# Load pretrained BERT model
model_name = 'bert-base-uncased'
model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: {model_name}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(model)

## 6. Tokenization

Convert text to token IDs using AutoTokenizer.

In [ ]:
# Example sentences
sentences = [
    "The transformer architecture revolutionized NLP.",
    "Attention is all you need.",
    "BERT learns contextual embeddings."
]

# Tokenize
for sent in sentences:
    tokens = tokenizer.tokenize(sent)
    token_ids = tokenizer.encode(sent, return_tensors='pt')
    print(f"\nSentence: {sent}")
    print(f"Tokens: {tokens}")
    print(f"Token IDs: {token_ids}")
    print(f"Num tokens: {token_ids.shape[1]}")

## 7. Extract BERT Embeddings

Generate embeddings and compute cosine similarity between sentences.

In [ ]:
model.eval()  # Set to evaluation mode

# Encode sentences
embeddings = []
for sent in sentences:
    inputs = tokenizer(sent, return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
        # Use [CLS] token embedding (first token)
        cls_embedding = outputs.last_hidden_state[:, 0, :].numpy()
        embeddings.append(cls_embedding[0])

embeddings = np.array(embeddings)
print(f"Embeddings shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")

# Compute cosine similarity
similarity = cosine_similarity(embeddings)

print("\nCosine Similarity Matrix:")
for i, sent in enumerate(sentences):
    print(f"\n{i}. {sent}")
    for j, sim in enumerate(similarity[i]):
        print(f"   vs {j}: {sim:.4f}")

## 8. Attention Visualization

Extract and visualize attention weights from BERT.

In [ ]:
# Get a sentence and extract attention weights
test_sent = "The transformer architecture revolutionized NLP."
inputs = tokenizer(test_sent, return_tensors='pt', padding=True, truncation=True)

with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)
    # attentions is a tuple of length = number of layers
    attentions = outputs.attentions

# Get attention from first layer, first head
attention_layer_0 = attentions[0][0, 0]  # [seq_len, seq_len]
tokens = tokenizer.tokenize(test_sent)

print(f"Sentence: {test_sent}")
print(f"Tokens: {tokens}")
print(f"Attention weights shape: {attention_layer_0.shape}")

# Visualize
plt.figure(figsize=(10, 8))
plt.imshow(attention_layer_0.numpy(), cmap='viridis')
plt.colorbar(label='Attention Weight')
plt.xticks(range(len(tokens)), tokens, rotation=45, ha='right')
plt.yticks(range(len(tokens)), tokens)
plt.xlabel('Key (Attended to)')
plt.ylabel('Query (Attending from)')
plt.title('BERT Attention Weights - Layer 0, Head 0')
plt.tight_layout()
plt.show()

## 9. Transformer Variants Comparison

Comparison of different Transformer architectures.

In [ ]:
comparison_data = {
    'Architecture': ['Encoder-only', 'Decoder-only', 'Encoder-Decoder'],
    'Self-Attention': ['Yes', 'Yes', 'Yes'],
    'Cross-Attention': ['No', 'No', 'Yes'],
    'Causal Masking': ['No', 'Yes', 'No'],
    'Use Cases': ['Classification, NER, Embeddings', 'Text generation, Language modeling', 'Translation, Summarization'],
    'Examples': ['BERT, RoBERTa, DistilBERT', 'GPT, GPT-2, Llama', 'T5, BART, mBART']
}

# Display as table
import pandas as pd
df = pd.DataFrame(comparison_data)
print(df.to_string(index=False))

## Interview Takeaways

### Key Concepts:

1. **Scaled Dot-Product Attention**: Core mechanism with query, key, value projections

2. **Multi-Head Attention**: Parallel attention heads for diverse representation subspaces

3. **Positional Encoding**: Sinusoidal encoding preserves position information (non-recurrent)

4. **Transformer Block**: Multi-head attention + Feed-forward with layer norm and residual connections

5. **Complexity**: O(n²) attention complexity on sequence length

6. **Parallelization**: All positions processed simultaneously (unlike RNNs)

7. **Variants**: Encoder (BERT), Decoder (GPT), Encoder-Decoder (T5)

8. **Pre-training**: Language modeling objectives learn general contextual representations

9. **Transfer Learning**: Fine-tune pretrained models for downstream tasks

10. **Attention Visualization**: Interpretability tool to understand what model attends to

### Common Interview Questions:

- What is the advantage of attention over RNNs? → Parallelization, long-range dependencies

- Why scale by √d_k? → Prevents softmax saturation, stabilizes gradients

- What is causal masking? → Prevents attending to future tokens in decoder

- How does BERT differ from GPT? → Bidirectional vs unidirectional, MLM vs CLM

- What is knowledge distillation? → Compress large models to smaller ones

- Explain transfer learning with transformers → Pre-train on large corpus, fine-tune on task

- What are limitations of transformers? → Memory usage, quadratic complexity, long sequences

- How to handle long sequences? → Sparse attention, local attention, linformer, longformer

- Explain attention head specialization → Different heads focus on different linguistic phenomena

- What is the role of FFN in transformers? → Non-linearity, feature expansion/compression


---

<small><em>© 2026 AI Nirvana · More Info: https://medium.com/@snigam/a-simple-structured-way-to-prepare-for-ai-ml-interviews-68b2e5830195 · Disclaimer: Provided as is. No liability assumed.</em></small>